# Convert AnnData to Seurat objects for interoperability

In [1]:
suppressPackageStartupMessages({
  library(Seurat)
  library(future)
  library(tidyverse)
})

In [ ]:
options(future.globals.maxSize = 1000000 * 1024^2, hpc.ncpus = 16)
plan(multicore, workers = getOption("hpc.ncpus", 1))
reticulate::use_condaenv(condaenv = "ocrelizumab_paper")

# Helper functions

In [3]:
source("../scripts/python.R")

Loading required package: reticulate



# Cantoni et al. (2025)

In [ ]:
seu <- h5ad.to.seurat(
  file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc.h5ad",
  x = "scale.data",
  raw = "counts",
  obsm = list(
    pca = "X_pca",
    integrated.rna = "X_pca_harmony"
  ),
  assay.name = "RNA"
) %>%
  NormalizeData(assay = "RNA")
seu$cell_type[which(seu$cell_type == "\u03b3\u03b4 T cells")] <- "gd T cells" # fix unicode characters

Normalizing layer: counts



In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "12.6 Gb"

In [ ]:
seu

An object of class Seurat 
18841 features across 184296 samples within 1 assay 
Active assay: RNA (18841 features, 0 variable features)
 3 layers present: counts, scale.data, data
 2 dimensional reductions calculated: pca, integrated.rna

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/cantoni/annotated/cantoni_untreated_ms_hc.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

# Absinta et al. (2021) and Lerma-Martin et al. (2024)

In [ ]:
seu <- h5ad.to.seurat(
  file = "../data/processed/external/lesion_rims/annotated/lesion_rims_lymphocytes.h5ad",
  x = "scale.data",
  raw = "counts",
  obsm = list(
    pca = "X_pca",
    integrated.rna = "X_pca_harmony"
  ),
  assay.name = "RNA"
) %>%
  NormalizeData(assay = "RNA")

Normalizing layer: counts



In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "0.1 Gb"

In [ ]:
seu

An object of class Seurat 
20521 features across 1517 samples within 1 assay 
Active assay: RNA (20521 features, 0 variable features)
 3 layers present: counts, scale.data, data
 2 dimensional reductions calculated: pca, integrated.rna

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/lesion_rims/annotated/lesion_rims_lymphocytes.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

# Kaufmann et al. (2021)

In [ ]:
seu <- h5ad.to.seurat(
  file = "../data/processed/external/kaufmann/annotated/kaufmann_full_rna.h5ad",
  x = "scale.data",
  raw = "counts",
  obsm = list(
    pca = "X_pca",
    integrated.rna = "X_pca_harmony"
  ),
  assay.name = "RNA"
) %>%
  NormalizeData(assay = "RNA")

Normalizing layer: counts



In [ ]:
oplan <- plan(sequential) # avoid parallelised CLR normalisation which is slower due to overheads with relatively small ADT panel
tmp <- h5ad.to.seurat(
  file = "../data/processed/external/kaufmann/annotated/kaufmann_full_adt.h5ad",
  x = "scale.data",
  raw = "counts",
  obsm = list(
    apca = "X_pca",
    integrated.adt = "X_pca_harmony"
  ),
  assay.name = "ADT"
) %>%
  split(f = .$sample_10X) %>%
  NormalizeData(assay = "ADT", normalization.method = "CLR", margin = 2) %>%
  JoinLayers(assay = "ADT", layers = c("counts", "data", "scale.data"))
for (x in c(Assays(tmp), Reductions(tmp))) {
  seu[[x]] <- tmp[[x]]
}
rm(tmp)
plan(oplan) # reset parallelisation strategy

Normalizing layer: counts.HH-OX-01_1

Normalizing across cells

Normalizing layer: counts.HH-OX-02_1

Normalizing across cells

Normalizing layer: counts.HH-OX-03_1

Normalizing across cells

Normalizing layer: counts.HH-OX-04_1

Normalizing across cells

Normalizing layer: counts.HH-OX-05_1

Normalizing across cells

Normalizing layer: counts.HH-OX-06_1

Normalizing across cells

Normalizing layer: counts.HH-OX-07_1

Normalizing across cells

Normalizing layer: counts.HH-OX-08_1

Normalizing across cells

Normalizing layer: counts.HH-OX-09_1

Normalizing across cells

Normalizing layer: counts.HH-OX-10_1

Normalizing across cells

Normalizing layer: counts.HH-OX-11_1

Normalizing across cells

Normalizing layer: counts.HH-OX-12_1

Normalizing across cells

Normalizing layer: counts.HH-OX-13_1

Normalizing across cells

Normalizing layer: counts.HH-OX-14_1

Normalizing across cells

Normalizing layer: counts.HH-OX-17_1

Normalizing across cells

Normalizing layer: counts.HH-OX-18_1

No

In [ ]:
object.size(seu) %>% format(unit = "Gb")

[1] "31.6 Gb"

In [ ]:
seu

An object of class Seurat 
15212 features across 497705 samples within 2 assays 
Active assay: RNA (15174 features, 0 variable features)
 3 layers present: counts, scale.data, data
 1 other assay present: ADT
 4 dimensional reductions calculated: pca, integrated.rna, apca, integrated.adt

In [ ]:
qs::qsave(
  seu,
  file = "../data/processed/external/kaufmann/annotated/kaufmann_full.qs",
  nthreads = getOption("hpc.ncpus", 1)
)

In [4]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /ceph/project/fuggerlab/rfarooq/.conda/envs/sandbox/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
[1] C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] reticulate_1.39.0  lubridate_1.9.3    forcats_1.0.0      stringr_1.5.1     
 [5] dplyr_1.1.4        purrr_1.0.2        readr_2.1.5        tidyr_1.3.1       
 [9] tibble_3.2.1       ggplot2_3.5.1      tidyverse_2.0.0    future_1.34.0     
[13] Seurat_5.1.0       SeuratObject_5.0.2 sp_2.1-4          

loaded via a namespace (and not attached):
  [1] deldir_2.0-4           pbapply_1.7-2          gridExtra_2.3         
  [4] rlang_1.1.4            magrittr_2.0.3         RcppAnnoy_0.0.22      
  [7] spatstat.geom_3.2-9    matrixStats_1.4.1      ggrid